# H-JEPA under dynamics shift: the "before" benchmark (Kaggle, 2× T4)

This notebook measures how the **frozen, released** three-level H-JEPA (FourRoom, `fourroom_hjepa_l3`) behaves when the environment's dynamics change at test time. No TTT and no looping run here: this is the baseline that the "after" notebook (JEPA-TTT, looped planner) is compared against, on the same tasks, the same shifted trajectories and the same metrics.

**What a shift is.** Only how a commanded action moves the agent changes. Rendering, walls, doors, distractors, goals and the 50 fixed evaluation tasks are unchanged. That isolates the part of the system JEPA-TTT adapts (the predictors), since the encoder sees nothing new.

**What it measures, and why each matters later**

| Part | Measures | What it tells the "after" work |
|---|---|---|
| A. Planning (50 tasks per shift) | success, final distance to goal, steps; paired per task against the clean run | the gap TTT or looping has to close |
| B. Model error per level (offline, on shifted trajectories) | 1-step loss, k-step rollout error, copy-last-latent baseline, shuffled-action gap | *which level's dynamics* the shift breaks, i.e. the target for JEPA-TTT |
| C. Representation probe | latent → agent (x, y), fit on clean, tested on shifted | that the encoder is unaffected (a dynamics shift should not change it) |
| D. Planner headroom (optional) | the same tasks with up to 3× planner iterations | whether better optimisation alone helps, i.e. the target for a looped planner |

**Before running (Kaggle):** *Settings → Accelerator → GPU T4 ×2*, *Settings → Internet → On*. Then *Run all*, or *Save Version → Save & Run All* to run headless (Kaggle allows 12 h per session; the default settings need roughly 4 to 6 h, estimated from the ~65 s/task we measured on one T4).

**Resumable:** every task and every condition writes its own result file and is skipped when it exists, so re-running a cell continues where it stopped (within the same session's `/kaggle/working`).

Pinned code: `kevinghst/H-JEPA` at `97054ec`.

## 0. Settings
Edit this cell only.

In [ ]:
SEEDS = [42]                # model seeds (the planner seed is paired with it, as in the paper); add 43, 44 for error bars across seeds

# Dynamics shifts. 'world' overrides arguments of the FourRoom env; 'shift' transforms the commanded action.
#   rotate: constant rotation of the action by `deg`
#   wave:   rotation amp_deg*sin(2*pi*x/period_px) that depends on the agent's x position (JEPA-TTT's Two-Room field)
CONDITIONS = {
    'clean':    {},                                                     # reference; must reproduce ~96%
    'speed_x2': {'world': {'agent_speed': 5.0}},                        # training speed is 2.5
    'rot45':    {'shift': {'type': 'rotate', 'deg': 45}},
    'rot90':    {'shift': {'type': 'rotate', 'deg': 90}},
    'wave75':   {'shift': {'type': 'wave', 'amp_deg': 75, 'period_px': 112}},
}

# D. Planner headroom: same tasks, planner allowed 90..270 GD iterations per solve instead of <=90
#    (early stopping kept, but not before 90). Paired per task with the 1x run of the same shift.
RUN_HEADROOM = True
HEADROOM_TASKS = 20
HEADROOM_SHIFTS = ['clean', 'rot45']
GD3X = {f'level{l}': {'n_steps': 270, 'min_its': 90} for l in (1, 2, 3)}

COLLECT_EPISODES = 100      # shifted trajectories per condition for part B (400 steps each, the dataset's own behaviour policy)
MODEL_ERROR_CLIPS = 1024    # clips per condition for part B
SMOKE = True                # run one short task per condition first (checks everything, prints time per task)

REPO_URL = 'https://github.com/kevinghst/H-JEPA.git'
REPO_COMMIT = '97054ecaff83e13839195ec4b92d74c08a0127c5'

## 1. GPUs and helpers

In [ ]:
import subprocess, sys, os, json, time, pathlib, shutil

def run(cmd, cwd=None, env=None, check=True):
    # Run a shell command, stream its output live, raise if it fails.
    print(f'$ {cmd}', flush=True)
    p = subprocess.Popen(cmd, shell=True, cwd=cwd, env=env, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='', flush=True)
    p.wait()
    if check and p.returncode != 0:
        raise RuntimeError(f'command failed with exit code {p.returncode}: {cmd}')
    return p.returncode

out = subprocess.run('nvidia-smi --query-gpu=index,name,memory.total --format=csv,noheader',
                     shell=True, capture_output=True, text=True)
assert out.returncode == 0 and out.stdout.strip(), 'No GPU. Settings -> Accelerator -> GPU T4 x2.'
GPUS = [int(l.split(',')[0]) for l in out.stdout.strip().splitlines()]
print(out.stdout.strip())
print('using GPUs', GPUS)

## 2. Paths
Code, environment and downloads go to local disk (`/tmp`); results and the shifted trajectories go to `/kaggle/working`, which Kaggle keeps as the notebook's output.

In [ ]:
KAGGLE = os.path.isdir('/kaggle/working')
WORK = '/kaggle/working' if KAGGLE else os.path.abspath('work')
SCRATCH = '/tmp/hjepa'
HJEPA_HOME = f'{SCRATCH}/hjepa_home'          # checkpoints + evaluation tasks
REPO = f'{SCRATCH}/H-JEPA'
H = f'{REPO}/h_jepa'
VENV = f'{SCRATCH}/venv'
PY = f'{VENV}/bin/python'
OUT = f'{WORK}/shift_bench'                   # results (keep this folder: the "after" notebook reads it)
DATA = f'{OUT}/data'                          # shifted trajectories (HDF5, same format as the H-JEPA datasets)
for d in (SCRATCH, HJEPA_HOME, OUT, DATA, f'{OUT}/logs'):
    os.makedirs(d, exist_ok=True)

# SDL_VIDEODRIVER: headless pygame rendering. MPLBACKEND: notebook kernels set an inline backend
# that does not exist inside the venv. One torch thread pool per process keeps 2 GPU workers from fighting over 4 CPUs.
ENV = dict(os.environ, HJEPA_HOME=HJEPA_HOME, SDL_VIDEODRIVER='dummy', MUJOCO_GL='egl', PYOPENGL_PLATFORM='egl',
           MPLBACKEND='Agg', PYTHONUNBUFFERED='1', HYDRA_FULL_ERROR='1', OMP_NUM_THREADS='2', MKL_NUM_THREADS='2')
print('results ->', OUT)

## 3. Code (pinned commit)

In [ ]:
if not os.path.isdir(f'{REPO}/.git'):
    run(f'git clone {REPO_URL} {REPO}')
run(f'git -C {REPO} checkout -q {REPO_COMMIT}')
head = subprocess.run(f'git -C {REPO} rev-parse HEAD', shell=True, capture_output=True, text=True).stdout.strip()
assert head == REPO_COMMIT, f'unexpected commit {head}'
print('code at', head)

## 4. Install (about 5 to 10 minutes)
The repo needs **Python 3.11 exactly** (`torch==2.7.1`). Kaggle's own Python may differ, so a separate 3.11 environment is built with `uv` and all repo code runs inside it.

In [ ]:
run('apt-get -qq update && apt-get -qq install -y libegl1 libgl1 libglu1-mesa > /dev/null', check=False)
run(f'{sys.executable} -m pip install -q uv')
if not os.path.exists(PY):
    run(f'uv venv --python 3.11 {VENV}')
run(f'uv pip install --python {PY} -e ".[train,env]"', cwd=REPO)
run(f'uv pip install --python {PY} "huggingface_hub>=0.34"')

## 5. Environment checks (every GPU)

In [ ]:
CHECK_ENV = r'''
import sys, torch
assert sys.version_info[:2] == (3, 11), sys.version
assert torch.__version__.startswith('2.7.1'), torch.__version__
assert torch.cuda.is_available(), 'torch cannot see the GPU'
cap = torch.cuda.get_device_capability(); arch = f'sm_{cap[0]}{cap[1]}'
assert arch in torch.cuda.get_arch_list(), f'no kernels for {arch}'
x = torch.randn(512, 512, device='cuda', requires_grad=True); (x @ x).sum().backward()
assert torch.isfinite(x.grad).all()
import matplotlib, hydra, sklearn, h5py, hdf5plugin, stable_worldmodel
print('ok:', torch.cuda.get_device_name(), arch, '| python', sys.version.split()[0], '| torch', torch.__version__)
'''
for g in GPUS:
    run(f'{PY} -c "$CHECK_ENV"', env=dict(ENV, CHECK_ENV=CHECK_ENV, CUDA_VISIBLE_DEVICES=str(g)), cwd=H)

## 6. Download checkpoints and evaluation tasks
From the Hugging Face repos `jepa-world-models/h-jepa` (model and dataset). Task files are verified against the published SHA-256 sums.

In [ ]:
for s in SEEDS:
    assert s in (42, 43, 44), s
ckpt_patterns = [f'fourroom/fourroom_hjepa_l3/seed{s}/*' for s in SEEDS]
DOWNLOAD = f'''
from huggingface_hub import snapshot_download
snapshot_download('jepa-world-models/h-jepa', repo_type='model', local_dir='{HJEPA_HOME}/ckpts', allow_patterns={ckpt_patterns!r})
snapshot_download('jepa-world-models/h-jepa', repo_type='dataset', local_dir='{HJEPA_HOME}',
                  allow_patterns=['eval_trajs/fourroom/*', 'eval_trajs/SHA256SUMS'])
print('download done')
'''
run(f'{PY} -c "$DOWNLOAD"', env=dict(ENV, DOWNLOAD=DOWNLOAD))
run('sha256sum -c --ignore-missing SHA256SUMS', cwd=f'{HJEPA_HOME}/eval_trajs')
assets = pathlib.Path(f'{H}/assets'); assets.mkdir(exist_ok=True)
link = assets / 'eval_trajs'
if link.is_symlink() or link.exists():
    assert link.is_symlink(), f'{link} exists and is not a symlink'
    link.unlink()
link.symlink_to(f'{HJEPA_HOME}/eval_trajs')
assert (link / 'fourroom/fourroom_tp35_cross2_goal75_n50_d1.pt').exists()

def ckpt(seed):
    return f'{HJEPA_HOME}/ckpts/fourroom/fourroom_hjepa_l3/seed{seed}/fourroom_hjepa_l3_object.ckpt'

for s in SEEDS:
    assert os.path.isfile(ckpt(s)) and os.path.isfile(os.path.join(os.path.dirname(ckpt(s)), 'normalizer.pt')), ckpt(s)
    print('OK', ckpt(s), f'{os.path.getsize(ckpt(s))/1e6:.0f} MB')

## 7. Write the benchmark script and the condition list
The whole benchmark lives in one script, `hjepa_shift_bench.py` (subcommands `collect`, `model-error`, `plan`, `summarize`). A copy is saved next to the results so the "after" notebook runs the identical code.

In [ ]:
BENCH_SRC = '"""H-JEPA FourRoom dynamics-shift benchmark ("before": frozen model, stock planner).\n\nRun from the repo\'s h_jepa/ directory with the repo\'s Python 3.11 environment.\n\nSubcommands\n  collect      record shifted trajectories with the dataset\'s own collection policy (CPU)\n  model-error  per-level latent prediction error of the frozen model on those trajectories (GPU)\n  plan         closed-loop planning on the 50 fixed tasks under each shift, sharded over GPUs\n  summarize    aggregate everything into summary.json (+ paired tests against the clean condition)\n\nEvery condition is a dynamics shift only: what the agent sees (rendering, distractors, goals,\ntasks) is unchanged, only how a commanded action moves the agent. The "after" benchmark\n(JEPA-TTT, looped planner) reuses this file unchanged and only swaps the model or planner, so the\ntwo are compared on identical tasks, trajectories and metrics.\n"""\nimport argparse\nimport json\nimport math\nimport os\nimport sys\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\n\nsys.path.insert(0, os.getcwd())  # h_jepa/ modules (data, planning_eval, the pickled model classes)\n\nBENCH_VERSION = 1\nENV_ID = \'swm/FourRoomShift-v0\'\n\n# --------------------------------------------------------------------------------------\n# Shifted environment\n# --------------------------------------------------------------------------------------\n\n\ndef _rotate(a, ang):\n    c, s = math.cos(ang), math.sin(ang)\n    return np.array([c * a[0] - s * a[1], s * a[0] + c * a[1]], dtype=np.float32)\n\n\ndef make_shift_env_class():\n    from stable_worldmodel.envs.four_room.env import FourRoomDistractorsEnv\n\n    class FourRoomShiftEnv(FourRoomDistractorsEnv):\n        """FourRoomDistractors with a test-time change to the action -> motion map.\n\n        shift = {\'type\': \'none\'}\n              | {\'type\': \'rotate\', \'deg\': d}                       constant rotation of the commanded action\n              | {\'type\': \'wave\', \'amp_deg\': A, \'period_px\': P}     rotation A*sin(2*pi*x/P) that depends on the\n                                                                   agent\'s x position (JEPA-TTT\'s Two-Room field)\n        Agent speed is changed with the base env\'s own `agent_speed` argument.\n        The commanded action is clipped to [-1, 1] first (as the base env does), then transformed;\n        the base env clips again after the transform.\n        """\n\n        def __init__(self, *args, shift=None, **kwargs):\n            self.shift = dict(shift or {\'type\': \'none\'})\n            assert self.shift.get(\'type\', \'none\') in (\'none\', \'rotate\', \'wave\'), self.shift\n            super().__init__(*args, **kwargs)\n\n        def transform_action(self, action):\n            a = np.clip(np.asarray(action, dtype=np.float32).reshape(-1), -1.0, 1.0)\n            kind = self.shift.get(\'type\', \'none\')\n            if kind == \'none\':\n                return a\n            if kind == \'rotate\':\n                return _rotate(a, math.radians(float(self.shift[\'deg\'])))\n            x = float(self.agent_position[0])\n            ang = math.radians(float(self.shift[\'amp_deg\'])) * math.sin(\n                2.0 * math.pi * x / float(self.shift[\'period_px\']))\n            return _rotate(a, ang)\n\n        def step(self, action):\n            return super().step(self.transform_action(action))\n\n    return FourRoomShiftEnv\n\n\ndef register_shift_env():\n    import gymnasium as gym\n    if ENV_ID not in gym.registry:\n        gym.register(id=ENV_ID, entry_point=make_shift_env_class())\n\n\n# --------------------------------------------------------------------------------------\n# Conditions\n# --------------------------------------------------------------------------------------\n\n\ndef load_conditions(out):\n    return json.loads((Path(out) / \'conditions.json\').read_text())\n\n\ndef world_overrides(cond):\n    over = dict(cond.get(\'world\', {}))\n    over[\'env_name\'] = ENV_ID\n    over[\'shift\'] = cond.get(\'shift\', {\'type\': \'none\'})\n    return over\n\n\ndef _json_default(o):\n    if isinstance(o, (np.integer,)):\n        return int(o)\n    if isinstance(o, (np.floating,)):\n        return float(o)\n    if isinstance(o, np.ndarray):\n        return o.tolist()\n    if torch.is_tensor(o):\n        return o.detach().cpu().tolist()\n    raise TypeError(type(o))\n\n\ndef write_json(path, obj):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_suffix(path.suffix + \'.tmp\')\n    tmp.write_text(json.dumps(obj, default=_json_default, indent=1))\n    os.replace(tmp, path)\n\n\n# --------------------------------------------------------------------------------------\n# collect: shifted trajectories with the dataset\'s own behaviour policy\n# --------------------------------------------------------------------------------------\n\n\ndef cmd_collect(args):\n    from omegaconf import OmegaConf\n    import stable_worldmodel as swm\n    from stable_worldmodel.envs.four_room import EpisodicPolicyMixture\n\n    register_shift_env()\n    repo = Path.cwd().parent  # run from h_jepa/\n    base = OmegaConf.load(repo / \'scripts/data/config/fourroom_tp35_d1_val.yaml\')\n    conds = load_conditions(args.out)\n    data_dir = Path(args.data_dir)\n    data_dir.mkdir(parents=True, exist_ok=True)\n\n    for name in args.conditions.split(\',\'):\n        cond = conds[name]\n        h5 = data_dir / f\'{name}.h5\'\n        done = data_dir / f\'{name}.done\'\n        if done.exists():\n            print(f\'[collect] {name}: already done\')\n            continue\n        if h5.exists():\n            h5.unlink()  # partial file from an interrupted run: start over (collection is deterministic)\n        world_cfg = OmegaConf.to_container(base.world, resolve=True)\n        world_cfg.update(world_overrides(cond))\n        world_cfg[\'num_envs\'] = int(args.num_envs)\n        world_cfg[\'terminate_on_goal\'] = False\n        world_cfg[\'retarget_on_goal\'] = True\n        env_name = world_cfg.pop(\'env_name\')\n        world = swm.World(env_name, **world_cfg, render_mode=\'rgb_array\')\n        rng = np.random.default_rng(int(args.seed))\n        policy = EpisodicPolicyMixture.from_config(\n            base.ego_policy.mixture, total_episodes=int(args.episodes),\n            seed=rng.integers(0, 1_000_000).item())\n        world.set_policy(policy)\n        t0 = time.time()\n        world.record_dataset(name, episodes=int(args.episodes), seed=rng.integers(0, 1_000_000).item(),\n                             cache_dir=str(data_dir))\n        world.close()\n        write_json(data_dir / f\'{name}.collection.json\', {\n            \'condition\': cond, \'episodes\': int(args.episodes), \'seed\': int(args.seed),\n            \'max_episode_steps\': int(world_cfg[\'max_episode_steps\']), \'seconds\': time.time() - t0,\n            \'policy_mixture\': OmegaConf.to_container(base.ego_policy.mixture, resolve=True)})\n        done.write_text(\'ok\')\n        print(f\'[collect] {name}: {args.episodes} episodes in {time.time() - t0:.0f}s -> {h5}\')\n\n\n# --------------------------------------------------------------------------------------\n# model-error: per-level latent prediction error of the frozen model\n# --------------------------------------------------------------------------------------\n\nHIST = 3  # predictor context (ARPredictor.pos_embedding has 3 positions)\n\n\ndef _load_model(policy, device):\n    model = torch.load(policy, map_location=\'cpu\', weights_only=False)\n    model = model.to(device).eval()\n    for p in model.parameters():\n        p.requires_grad_(False)\n    return model\n\n\ndef _level_geometry(model):\n    geo = {}\n    for lvl in range(1, model.num_levels + 1):\n        j = model.get_level(lvl)\n        geo[lvl] = (int(getattr(j, \'temporal_stride\', 1)), int(getattr(j, \'temporal_window_size\', 1)))\n    return geo\n\n\ndef _level1_frames_needed(geo, top_steps):\n    need = top_steps\n    for lvl in range(len(geo), 1, -1):\n        stride, window = geo[lvl]\n        need = (need - 1) * stride + window\n    return need\n\n\ndef _xy_for_level(out, lvl):\n    v = out.get(f\'xy_level{lvl}\')\n    if v is None:\n        return None\n    if v.ndim == 4:  # chunked windows (B, T, w, 2): the window\'s last state\n        v = v[:, :, -1]\n    return v\n\n\n@torch.no_grad()\ndef _encode_batch(model, batch, device):\n    from data import IMAGENET_STATS\n    b = {k: (v.to(device) if torch.is_tensor(v) else v) for k, v in batch.items()}\n    px = b[\'pixels_level1\']\n    if px.dtype == torch.uint8:\n        mean = torch.tensor(IMAGENET_STATS[\'mean\'], device=device).view(1, 1, 3, 1, 1)\n        std = torch.tensor(IMAGENET_STATS[\'std\'], device=device).view(1, 1, 3, 1, 1)\n        b[\'pixels_level1\'] = px.float().div(255.0).sub(mean).div(std)\n    saved = [model.get_level(l).target_length for l in range(1, model.num_levels + 1)]\n    try:\n        for l in range(1, model.num_levels + 1):\n            model.get_level(l).target_length = None  # keep full sequences (no random crop)\n        out = model.encode_hierarchical(b, sparse_level1_encode=False)\n    finally:\n        for l, t in zip(range(1, model.num_levels + 1), saved):\n            model.get_level(l).target_length = t\n    return out\n\n\n@torch.no_grad()\ndef _ar_errors(jepa, emb, act, ks, perm=None):\n    """Autoregressive k-step prediction from the true context ending at index HIST-1.\n\n    Position j of the predictor output predicts emb[j+1] from emb[..j], act[..j] (training\n    alignment). Returns dict k -> per-sample MSE (mean over D), cosine and norm ratio.\n    """\n    s = HIST - 1\n    kmax = max(ks)\n    a = act if perm is None else act[perm]\n    seq = emb[:, :HIST]\n    preds = []\n    for k in range(kmax):\n        p = jepa.predict(seq[:, -HIST:], a[:, k:k + HIST])[:, -1]\n        preds.append(p)\n        seq = torch.cat([seq, p[:, None]], dim=1)\n    res = {}\n    for k in ks:\n        p, t = preds[k - 1], emb[:, s + k]\n        res[k] = {\n            \'mse\': ((p - t) ** 2).mean(-1),\n            \'cos\': torch.nn.functional.cosine_similarity(p, t, dim=-1),\n            \'norm_ratio\': p.norm(dim=-1) / t.norm(dim=-1).clamp_min(1e-8),\n        }\n    return res\n\n\ndef _ridge_fit(x, y, lam=1e-2):\n    x1 = np.concatenate([x, np.ones((len(x), 1))], 1)\n    reg = lam * np.eye(x1.shape[1])\n    reg[-1, -1] = 0\n    return np.linalg.solve(x1.T @ x1 + reg, x1.T @ y)\n\n\ndef _ridge_r2(w, x, y):\n    x1 = np.concatenate([x, np.ones((len(x), 1))], 1)\n    pred = x1 @ w\n    ss_res = ((y - pred) ** 2).sum(0)\n    ss_tot = ((y - y.mean(0)) ** 2).sum(0)\n    return float(np.mean(1 - ss_res / np.maximum(ss_tot, 1e-12))), float(np.sqrt(((y - pred) ** 2).sum(1).mean()))\n\n\ndef cmd_model_error(args):\n    import stable_worldmodel as swm\n    from data import Compose, get_column_normalizer_from_artifact\n\n    device = \'cuda\' if torch.cuda.is_available() else \'cpu\'\n    torch.manual_seed(0)\n    model = _load_model(args.policy, device)\n    artifact = model.normalizer_artifact\n    geo = _level_geometry(model)\n    L = model.num_levels\n    top_steps = HIST + int(args.top_k)\n    n1 = _level1_frames_needed(geo, top_steps)\n    level_cfgs = {\'level1\': {\'frameskip\': 5, \'num_steps\': 2}}  # FourRoom level-1 frameskip (train config)\n    for lvl in range(2, L + 1):\n        level_cfgs[f\'level{lvl}\'] = {\'frameskip\': geo[lvl][0], \'window_size\': geo[lvl][1],\n                                     \'num_steps\': top_steps if lvl == L else 2}\n    ks = {lvl: sorted({k for k in (1, 2, 4, 8) if k <= _steps_at_level(geo, lvl, n1) - HIST}) for lvl in range(1, L + 1)}\n    print(f\'[model-error] levels={L} geometry={geo} level-1 frames per clip={n1} horizons={ks}\')\n\n    out_dir = Path(args.out) / \'model_error\' / f\'seed{args.seed}\'\n    out_dir.mkdir(parents=True, exist_ok=True)\n    probe_cache = {}\n    for name in args.conditions.split(\',\'):\n        res_path = out_dir / f\'{name}.json\'\n        if res_path.exists() and not args.force:\n            print(f\'[model-error] {name}: exists\')\n            continue\n        ds = swm.data.HDF5Dataset(name, keys_to_load=[\'pixels\', \'action\', \'xy\'], keys_to_cache=[\'action\', \'xy\'],\n                                  cache_dir=args.data_dir, **level_cfgs)\n        ds.transform = Compose(get_column_normalizer_from_artifact(artifact, \'action\', \'action\'))\n        # clips start every span//5 raw steps (overlapping), evenly spread over episodes; standard\n        # errors below are clustered by episode, so the overlap does not overstate precision\n        span = ds.last_level[\'span\']\n        hop = max(1, span // 5)\n        clips = [(ep, st) for ep, st in ds.clip_indices if st % hop == 0]\n        rng = np.random.default_rng(0)\n        if len(clips) > args.max_clips:\n            clips = [clips[i] for i in sorted(rng.choice(len(clips), args.max_clips, replace=False))]\n        ds.clip_indices = clips\n        loader = torch.utils.data.DataLoader(ds, batch_size=args.batch_size, shuffle=False,\n                                             num_workers=args.num_workers)\n        acc = {lvl: {} for lvl in range(1, L + 1)}\n        feats = {lvl: [] for lvl in range(1, L + 1)}\n        xys = {lvl: [] for lvl in range(1, L + 1)}\n        g = torch.Generator(device=\'cpu\').manual_seed(0)\n        t0 = time.time()\n        for batch in loader:\n            out = _encode_batch(model, batch, device)\n            B = out[\'embed_1\'].size(0)\n            perm = torch.randperm(B, generator=g).to(device)\n            for lvl in range(1, L + 1):\n                jepa = model.get_level(lvl)\n                emb, act = out[f\'embed_{lvl}\'].float(), out[f\'action_{lvl}\'].float()\n                true = _ar_errors(jepa, emb, act, ks[lvl])\n                shuf = _ar_errors(jepa, emb, act, ks[lvl], perm=perm)\n                # teacher-forced one-step loss over every window (the training / JEPA-TTT loss)\n                T = emb.size(1)\n                tf = []\n                for st in range(0, T - HIST):\n                    p = jepa.predict(emb[:, st:st + HIST], act[:, st:st + HIST])\n                    tf.append(((p - emb[:, st + 1:st + 1 + HIST]) ** 2).mean((-1, -2)))\n                d = acc[lvl]\n                d.setdefault(\'tf_mse\', []).append(torch.stack(tf, 1).mean(1).cpu())\n                d.setdefault(\'emb_var\', []).append(emb.var(0).mean().repeat(B).cpu())\n                for k in ks[lvl]:\n                    d.setdefault(f\'mse_k{k}\', []).append(true[k][\'mse\'].cpu())\n                    d.setdefault(f\'cos_k{k}\', []).append(true[k][\'cos\'].cpu())\n                    d.setdefault(f\'norm_ratio_k{k}\', []).append(true[k][\'norm_ratio\'].cpu())\n                    d.setdefault(f\'shuffled_mse_k{k}\', []).append(shuf[k][\'mse\'].cpu())\n                    d.setdefault(f\'copy_mse_k{k}\', []).append(((emb[:, HIST - 1] - emb[:, HIST - 1 + k]) ** 2).mean(-1).cpu())\n                xy = _xy_for_level(out, lvl)\n                if xy is not None:\n                    feats[lvl].append(emb.reshape(-1, emb.size(-1)).cpu().numpy())\n                    xys[lvl].append(xy.reshape(-1, xy.size(-1)).float().cpu().numpy())\n        result = {\'condition\': name, \'seed\': args.seed, \'policy\': args.policy, \'n_clips\': len(clips),\n                  \'level1_frames_per_clip\': n1, \'horizons\': ks, \'seconds\': time.time() - t0,\n                  \'bench_version\': BENCH_VERSION, \'levels\': {}}\n        ep_of_clip = np.array([c[0] for c in clips])\n        arrays = {\'episode\': ep_of_clip}\n        for lvl in range(1, L + 1):\n            lv = {}\n            for key, parts in acc[lvl].items():\n                v = torch.cat(parts).numpy().astype(np.float64)\n                arrays[f\'L{lvl}_{key}\'] = v\n                lv[key] = {\'mean\': float(v.mean()), \'se_episode\': _cluster_se(v, ep_of_clip)}\n            var = lv[\'emb_var\'][\'mean\']\n            for k in ks[lvl]:\n                lv[f\'rel_mse_k{k}\'] = lv[f\'mse_k{k}\'][\'mean\'] / max(var, 1e-12)\n                lv[f\'action_gap_k{k}\'] = lv[f\'shuffled_mse_k{k}\'][\'mean\'] - lv[f\'mse_k{k}\'][\'mean\']\n                lv[f\'beats_copy_k{k}\'] = lv[f\'mse_k{k}\'][\'mean\'] < lv[f\'copy_mse_k{k}\'][\'mean\']\n            if feats[lvl]:\n                probe_cache[(name, lvl)] = (np.concatenate(feats[lvl]), np.concatenate(xys[lvl]))\n            result[\'levels\'][str(lvl)] = lv\n        np.savez_compressed(out_dir / f\'{name}.npz\', **arrays)\n        write_json(res_path, result)\n        print(f\'[model-error] {name}: {len(clips)} clips, {time.time() - t0:.0f}s\')\n        for lvl in range(1, L + 1):\n            lv = result[\'levels\'][str(lvl)]\n            k = ks[lvl][-1]\n            print(f\'    L{lvl}: tf_mse={lv["tf_mse"]["mean"]:.4f}  mse_k{k}={lv[f"mse_k{k}"]["mean"]:.4f} \'\n                  f\'copy_k{k}={lv[f"copy_mse_k{k}"]["mean"]:.4f}  action_gap_k{k}={lv[f"action_gap_k{k}"]:.4f}\')\n\n    # representation probe: latent -> agent xy, fit on clean (held-out episodes), tested on every condition\n    ref = args.probe_reference\n    if any(key[0] == ref for key in probe_cache) or (out_dir / f\'{ref}.json\').exists():\n        _probe(args, out_dir, probe_cache, L)\n\n\ndef _cluster_se(v, groups):\n    """Standard error of the mean with clips clustered by episode."""\n    means = np.array([v[groups == g].mean() for g in np.unique(groups)])\n    return float(means.std(ddof=1) / math.sqrt(len(means))) if len(means) > 1 else float(\'nan\')\n\n\ndef _steps_at_level(geo, lvl, n1):\n    n = n1\n    for l in range(2, lvl + 1):\n        stride, window = geo[l]\n        n = (n - window) // stride + 1\n    return n\n\n\ndef _probe(args, out_dir, cache, L):\n    ref = args.probe_reference\n    if not any(k[0] == ref for k in cache):\n        print(f\'[probe] reference condition {ref!r} was not encoded in this call; run model-error with it included\')\n        return\n    probe = {}\n    for lvl in range(1, L + 1):\n        if (ref, lvl) not in cache:\n            continue\n        x, y = cache[(ref, lvl)]\n        n = len(x)\n        cut = int(0.7 * n)  # clips are episode-ordered, so this is an episode-wise split\n        w = _ridge_fit(x[:cut], y[:cut])\n        lv = {\'reference\': ref, \'train_frames\': cut}\n        lv[f\'{ref}_heldout\'] = dict(zip((\'r2\', \'rmse_px\'), _ridge_r2(w, x[cut:], y[cut:])))\n        for (name, l2), (xc, yc) in cache.items():\n            if l2 == lvl and name != ref:\n                lv[name] = dict(zip((\'r2\', \'rmse_px\'), _ridge_r2(w, xc, yc)))\n        probe[str(lvl)] = lv\n        print(f\'[probe] L{lvl}: \' + \', \'.join(f\'{k}: R2={v["r2"]:.3f}\' for k, v in lv.items() if isinstance(v, dict)))\n    write_json(out_dir / \'xy_probe.json\', probe)\n\n\n# --------------------------------------------------------------------------------------\n# plan: closed-loop planning, sharded over GPUs\n# --------------------------------------------------------------------------------------\n\n_COST_STASH = {}\n\n\ndef _install_planner_recorders():\n    """Record each solve\'s final best cost (per level) next to the repo\'s own solve_records."""\n    import hierarchical_solver as hs\n    from stable_worldmodel.solver import gd\n\n    if getattr(hs.LevelCostModel, \'_bench_patched\', False):\n        return\n    orig_cost = hs.LevelCostModel.get_cost\n\n    def get_cost(self, info_dict, action_candidates):\n        costs = orig_cost(self, info_dict, action_candidates)\n        if not torch.is_grad_enabled():  # the solver\'s final no-grad evaluation\n            _COST_STASH[\'last\'] = costs.detach().min(dim=1).values.float().cpu().tolist()\n        return costs\n\n    hs.LevelCostModel.get_cost = get_cost\n    hs.LevelCostModel._bench_patched = True\n    orig_solve = gd.GradientSolver.solve\n\n    def solve(self, *a, **k):\n        _COST_STASH.pop(\'last\', None)\n        out = orig_solve(self, *a, **k)\n        if self.solve_records:\n            self.solve_records[-1][\'final_cost\'] = _COST_STASH.pop(\'last\', None)\n            self.solve_records[-1][\'n_steps_cap\'] = int(self.n_steps)\n        return out\n\n    gd.GradientSolver.solve = solve\n\n\ndef _build_cfg(args, cond):\n    from omegaconf import OmegaConf\n    from planning_eval import load_eval_config, _resolve_existing_path\n\n    cfg = load_eval_config(args.config_name)\n    cfg = OmegaConf.create(OmegaConf.to_container(cfg, resolve=False))\n    cfg.policy = args.policy\n    cfg.seed = int(args.seed)\n    cfg.eval.num_eval = int(cond.get(\'num_eval\', args.num_eval))\n    for k, v in world_overrides(cond).items():\n        cfg.world[k] = v\n    for lvl, over in cond.get(\'solver\', {}).items():\n        for k, v in over.items():\n            cfg.solver.solvers[lvl][k] = v\n    for item in args.override or []:  # generic key=value overrides (smoke tests), e.g. eval.eval_budget=30\n        key, val = item.split(\'=\', 1)\n        OmegaConf.update(cfg, key, _parse_value(val), force_add=True)\n    if not torch.cuda.is_available():\n        for lvl in cfg.solver.solvers:\n            cfg.solver.solvers[lvl].device = \'cpu\'\n    cfg.load_eval_trajs_path = str(_resolve_existing_path(cfg.load_eval_trajs_path))\n    return cfg\n\n\ndef _parse_value(val):\n    import yaml\n    return yaml.safe_load(val)\n\n\ndef cmd_plan(args):\n    from omegaconf import OmegaConf\n    from torch.nn.attention import SDPBackend, sdpa_kernel\n    import stable_worldmodel as swm\n    import planning_eval as pe\n\n    register_shift_env()\n    _install_planner_recorders()\n    conds = load_conditions(args.out)\n    torch.set_num_threads(max(1, int(args.threads)))\n\n    for name in args.conditions.split(\',\'):\n        cond = conds[name]\n        cfg = _build_cfg(args, cond)\n        cfg.world.max_episode_steps = 2 * cfg.eval.eval_budget\n        world_cfg = OmegaConf.create(OmegaConf.to_container(cfg.world, resolve=True))\n        world_cfg = OmegaConf.to_container(world_cfg)\n        transform = {\'pixels\': pe.img_transform(cfg), \'goal\': pe.img_transform(cfg)}\n        image_shape = (int(cfg.eval.img_size),) * 2\n        process = pe._build_eval_process_from_policy_normalizer(cfg, model=None)\n        payload = torch.load(cfg.load_eval_trajs_path, map_location=\'cpu\', weights_only=False)\n        n = int(cfg.eval.num_eval)\n        if args.limit is not None:\n            n = min(n, int(args.limit))\n        assert len(payload[\'data\']) >= n, f\'{len(payload["data"])} tasks < num_eval={n}\'\n        ep_idx = payload.get(\'episodes_idx\')\n        ep_idx = list(ep_idx) if ep_idx is not None and len(ep_idx) else list(range(len(payload[\'data\'])))\n        callables = OmegaConf.to_container(cfg.eval.get(\'callables\'), resolve=True)\n        res_dir = Path(args.out) / \'plan\' / name / f\'seed{args.seed}\'\n        (res_dir / \'tasks\').mkdir(parents=True, exist_ok=True)\n        if args.shard == 0:\n            (res_dir / \'eval_config.yaml\').write_text(OmegaConf.to_yaml(cfg, resolve=True))\n        my_tasks = [t for t in range(n) if t % args.num_shards == args.shard]\n        todo = [t for t in my_tasks if not (res_dir / \'tasks\' / f\'task_{t:03d}.json\').exists()]\n        print(f\'[plan] {name} seed{args.seed} shard {args.shard}/{args.num_shards}: \'\n              f\'{len(my_tasks) - len(todo)} done, {len(todo)} to go\', flush=True)\n        if not todo:\n            continue\n        policy = pe._build_policy(cfg, process, transform, model=None)\n        solver = getattr(policy, \'solver\', None)\n        gens = [(s.torch_gen, s.torch_gen.initial_seed())\n                for s in getattr(solver, \'level_solvers\', {1: solver}).values() if hasattr(s, \'torch_gen\')]\n        for t in todo:\n            for gen, seed in gens:  # identical to the repo\'s chunked eval with chunk_size=1\n                gen.manual_seed(seed + t)\n            chunk = {**payload, \'data\': [dict(payload[\'data\'][t])], \'episodes_idx\': ep_idx[t:t + 1]}\n            world = swm.World(**{**world_cfg, \'num_envs\': 1}, image_shape=image_shape)\n            trace = {\'dist\': [], \'xy\': []}\n            orig_step = world.step\n\n            def step_and_record(orig_step=orig_step, world=world, trace=trace):\n                orig_step()\n                info = world.infos\n                d = info.get(\'distance_to_target\')\n                xy = info.get(\'xy\')\n                if d is not None:\n                    trace[\'dist\'].append(float(np.asarray(d).reshape(-1)[0]))\n                if xy is not None:\n                    trace[\'xy\'].append(np.asarray(xy, dtype=np.float64).reshape(-1)[-2:].tolist())\n\n            world.step = step_and_record\n            try:\n                world.set_policy(policy)\n                t0 = time.time()\n                with torch.enable_grad(), sdpa_kernel(SDPBackend.MATH):\n                    m = world.evaluate_from_dataset(\n                        None, episodes_idx=None, start_steps=None,\n                        goal_offset_steps=cfg.eval.goal_offset_steps, eval_budget=cfg.eval.eval_budget,\n                        callables=callables, load_eval_trajs_path=chunk, process=process,\n                        start_state_mode=cfg.eval.get(\'start_state_mode\', \'dataset_full\'),\n                        goal_state_mode=cfg.eval.get(\'goal_state_mode\', \'dataset_full\'))\n                secs = time.time() - t0\n            finally:\n                world.close()\n            success = bool(np.asarray(m[\'episode_successes\']).reshape(-1)[0])\n            steps = float(np.asarray(m[\'steps_to_success\']).reshape(-1)[0])\n            dist = trace[\'dist\']\n            rec = {\n                \'task\': t, \'condition\': name, \'seed\': int(args.seed), \'success\': success,\n                \'steps_to_success\': None if not math.isfinite(steps) else steps,\n                \'final_distance\': dist[-1] if dist else None,\n                \'min_distance\': min(dist) if dist else None,\n                \'distance_trace\': dist, \'xy_trace\': trace[\'xy\'],\n                \'solve_records\': pe.pop_solve_records(solver),\n                \'seconds\': secs, \'bench_version\': BENCH_VERSION,\n            }\n            write_json(res_dir / \'tasks\' / f\'task_{t:03d}.json\', rec)\n            print(f\'[plan] {name} task {t:02d}: success={success} steps={rec["steps_to_success"]} \'\n                  f\'final_dist={rec["final_distance"]} ({secs:.0f}s)\', flush=True)\n\n\n# --------------------------------------------------------------------------------------\n# summarize\n# --------------------------------------------------------------------------------------\n\n\ndef wilson(k, n, z=1.96):\n    if n == 0:\n        return (float(\'nan\'), float(\'nan\'))\n    p = k / n\n    den = 1 + z * z / n\n    c = (p + z * z / (2 * n)) / den\n    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / den\n    return (100 * (c - h), 100 * (c + h))\n\n\ndef mcnemar_exact(b, c):\n    """Two-sided exact McNemar p-value from the discordant pair counts."""\n    n = b + c\n    if n == 0:\n        return 1.0\n    k = min(b, c)\n    p = sum(math.comb(n, i) for i in range(k + 1)) / 2 ** n\n    return min(1.0, 2 * p)\n\n\ndef _plan_summary(tasks, start_dist):\n    n = len(tasks)\n    succ = np.array([t[\'success\'] for t in tasks], dtype=bool)\n    k = int(succ.sum())\n    steps = [t[\'steps_to_success\'] for t in tasks if t[\'steps_to_success\'] is not None]\n    fin = np.array([t[\'final_distance\'] if t[\'final_distance\'] is not None else np.nan for t in tasks])\n    mind = np.array([t[\'min_distance\'] if t[\'min_distance\'] is not None else np.nan for t in tasks])\n    out = {\'n_tasks\': n, \'successes\': k, \'success_rate\': 100 * k / n if n else float(\'nan\'),\n           \'wilson95\': wilson(k, n),\n           \'mean_steps_success_only\': float(np.mean(steps)) if steps else None,\n           \'final_distance_mean\': float(np.nanmean(fin)) if n else None,\n           \'final_distance_failures_mean\': float(np.nanmean(fin[~succ])) if (~succ).any() else None,\n           \'min_distance_mean\': float(np.nanmean(mind)) if n else None,\n           \'seconds_per_task\': float(np.mean([t[\'seconds\'] for t in tasks])) if n else None}\n    if start_dist:\n        prog = [1 - (t[\'final_distance\'] / start_dist[t[\'task\']]) for t in tasks\n                if t[\'final_distance\'] is not None and start_dist.get(t[\'task\'])]\n        out[\'progress_mean\'] = float(np.mean(prog)) if prog else None\n    lv = {}\n    for t in tasks:\n        for r in t[\'solve_records\']:\n            d = lv.setdefault(str(r[\'level\']), {\'iters\': [], \'cap_hits\': 0, \'solves\': 0, \'final_cost\': [], \'wall\': []})\n            d[\'iters\'].extend(r[\'n_iters\'])\n            cap = r.get(\'n_steps_cap\')\n            d[\'cap_hits\'] += sum(int(cap is not None and it >= cap) for it in r[\'n_iters\'])\n            d[\'solves\'] += len(r[\'n_iters\'])\n            if r.get(\'final_cost\'):\n                d[\'final_cost\'].extend(r[\'final_cost\'])\n            d[\'wall\'].append(r[\'wall_clock\'])\n    out[\'planner\'] = {l: {\'solves\': d[\'solves\'], \'mean_iters\': float(np.mean(d[\'iters\'])) if d[\'iters\'] else None,\n                          \'cap_hit_frac\': d[\'cap_hits\'] / max(d[\'solves\'], 1),\n                          \'final_cost_mean\': float(np.mean(d[\'final_cost\'])) if d[\'final_cost\'] else None,\n                          \'wall_clock_total_s\': float(np.sum(d[\'wall\']))}\n                      for l, d in sorted(lv.items())}\n    return out\n\n\ndef cmd_summarize(args):\n    out = Path(args.out)\n    conds = load_conditions(out)\n    summary = {\'bench_version\': BENCH_VERSION, \'conditions\': conds, \'plan\': {}, \'model_error\': {}, \'probe\': {}}\n    ref = args.reference\n    plan_tasks = {}\n    for name in conds:\n        for seed_dir in sorted((out / \'plan\' / name).glob(\'seed*\')):\n            tasks = [json.loads(p.read_text()) for p in sorted((seed_dir / \'tasks\').glob(\'task_*.json\'))]\n            if tasks:\n                plan_tasks[(name, seed_dir.name)] = {t[\'task\']: t for t in tasks}\n    for (name, seed), tasks in plan_tasks.items():\n        start = {}\n        ref_tasks = plan_tasks.get((ref, seed), {})\n        for tid, t in ref_tasks.items():\n            if t[\'distance_trace\']:\n                start[tid] = t[\'distance_trace\'][0]\n        s = _plan_summary(list(tasks.values()), start)\n        bases = [b for b in (ref, conds[name].get(\'paired_with\')) if b and b != name]\n        for base in dict.fromkeys(bases):\n            base_tasks = plan_tasks.get((base, seed), {})\n            if not base_tasks:\n                continue\n            common = sorted(set(tasks) & set(base_tasks))\n            b = sum(base_tasks[i][\'success\'] and not tasks[i][\'success\'] for i in common)\n            c = sum((not base_tasks[i][\'success\']) and tasks[i][\'success\'] for i in common)\n            fd = [tasks[i][\'final_distance\'] - base_tasks[i][\'final_distance\'] for i in common\n                  if tasks[i][\'final_distance\'] is not None and base_tasks[i][\'final_distance\'] is not None]\n            s[\'paired_vs_\' + base] = {\n                \'n_common\': len(common),\n                \'success_this\': 100 * sum(tasks[i][\'success\'] for i in common) / max(len(common), 1),\n                \'success_base\': 100 * sum(base_tasks[i][\'success\'] for i in common) / max(len(common), 1),\n                \'delta_success_pts\': 100 * (sum(tasks[i][\'success\'] for i in common)\n                                            - sum(base_tasks[i][\'success\'] for i in common)) / max(len(common), 1),\n                \'lost\': b, \'gained\': c, \'mcnemar_p\': mcnemar_exact(b, c),\n                \'delta_final_distance_mean\': float(np.mean(fd)) if fd else None}\n        summary[\'plan\'].setdefault(name, {})[seed] = s\n    for p in sorted((out / \'model_error\').glob(\'seed*/*.json\')):\n        if p.name == \'xy_probe.json\':\n            summary[\'probe\'][p.parent.name] = json.loads(p.read_text())\n        else:\n            r = json.loads(p.read_text())\n            summary[\'model_error\'].setdefault(r[\'condition\'], {})[p.parent.name] = r\n    write_json(out / \'summary.json\', summary)\n    print(f\'wrote {out / "summary.json"}\')\n\n\n# --------------------------------------------------------------------------------------\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    sub = ap.add_subparsers(dest=\'cmd\', required=True)\n    c = sub.add_parser(\'collect\')\n    c.add_argument(\'--out\', required=True)\n    c.add_argument(\'--data-dir\', required=True)\n    c.add_argument(\'--conditions\', required=True)\n    c.add_argument(\'--episodes\', type=int, default=100)\n    c.add_argument(\'--num-envs\', type=int, default=10)\n    c.add_argument(\'--seed\', type=int, default=3072)\n    m = sub.add_parser(\'model-error\')\n    m.add_argument(\'--out\', required=True)\n    m.add_argument(\'--data-dir\', required=True)\n    m.add_argument(\'--conditions\', required=True)\n    m.add_argument(\'--policy\', required=True)\n    m.add_argument(\'--seed\', type=int, default=42)\n    m.add_argument(\'--top-k\', type=int, default=4, help=\'autoregressive horizon at the top level\')\n    m.add_argument(\'--max-clips\', type=int, default=1024)\n    m.add_argument(\'--batch-size\', type=int, default=16)\n    m.add_argument(\'--num-workers\', type=int, default=2)\n    m.add_argument(\'--probe-reference\', default=\'clean\')\n    m.add_argument(\'--force\', action=\'store_true\')\n    p = sub.add_parser(\'plan\')\n    p.add_argument(\'--out\', required=True)\n    p.add_argument(\'--conditions\', required=True)\n    p.add_argument(\'--policy\', required=True)\n    p.add_argument(\'--seed\', type=int, default=42)\n    p.add_argument(\'--config-name\', default=\'fourroom_l3\')\n    p.add_argument(\'--num-eval\', type=int, default=50)\n    p.add_argument(\'--shard\', type=int, default=0)\n    p.add_argument(\'--num-shards\', type=int, default=1)\n    p.add_argument(\'--threads\', type=int, default=2)\n    p.add_argument(\'--limit\', type=int, default=None, help=\'plan only the first N tasks of each condition\')\n    p.add_argument(\'--override\', action=\'append\', help=\'extra eval-config key=value (repeatable)\')\n    s = sub.add_parser(\'summarize\')\n    s.add_argument(\'--out\', required=True)\n    s.add_argument(\'--reference\', default=\'clean\')\n    args = ap.parse_args()\n    {\'collect\': cmd_collect, \'model-error\': cmd_model_error, \'plan\': cmd_plan, \'summarize\': cmd_summarize}[args.cmd](args)\n\n\nif __name__ == \'__main__\':\n    main()\n'

BENCH = f'{H}/hjepa_shift_bench.py'
pathlib.Path(BENCH).write_text(BENCH_SRC)
shutil.copy(BENCH, f'{OUT}/hjepa_shift_bench.py')

conds = {k: dict(v) for k, v in CONDITIONS.items()}
assert 'clean' in conds, "keep the 'clean' reference condition"
if RUN_HEADROOM:
    for base in HEADROOM_SHIFTS:
        assert base in conds, base
        conds[f'{base}_gd3x'] = {**conds[base], 'paired_with': base, 'num_eval': HEADROOM_TASKS, 'solver': GD3X}
cpath = pathlib.Path(f'{OUT}/conditions.json')
if cpath.exists() and json.loads(cpath.read_text()) != conds:
    raise RuntimeError(f'{cpath} holds a different condition list. Results in {OUT} were made with it; '
                       'use a new OUT folder (or delete the old one) before changing CONDITIONS.')
cpath.write_text(json.dumps(conds, indent=1))
SHIFTS = list(CONDITIONS)                  # conditions with their own trajectories (part B)
PLAN_CONDS = list(conds)                   # everything that is planned (parts A and D)
print(json.dumps(conds, indent=1))

## 8. Check the shifts do exactly what they say
For every condition, the agent is placed in open space and given unit actions; the measured displacement must equal `speed × R(θ) × action`. The `clean` condition must also reproduce the stock environment step for step.

In [ ]:
CHECK_SHIFT = r'''
import json, math, os, sys
import numpy as np, torch, gymnasium as gym
from omegaconf import OmegaConf
sys.path.insert(0, '.')
import hjepa_shift_bench as hb
hb.register_shift_env()
conds = hb.load_conditions(os.environ['OUT'])
base = OmegaConf.to_container(OmegaConf.load('config/eval/fourroom_l3.yaml').world, resolve=False)
for k in ('num_envs', 'max_episode_steps', 'history_size', 'frame_skip', 'env_name'):
    base.pop(k)
for name in json.loads(os.environ['SHIFTS']):
    c = conds[name]
    kw = dict(base, **c.get('world', {}), shift=c.get('shift', {'type': 'none'}))
    env = gym.make(hb.ENV_ID, **kw).unwrapped
    env.reset(seed=0)
    speed = float(env.variation_space['agent']['speed'].value.item())
    for pos in ([56.0, 56.0], [168.0, 56.0], [80.0, 168.0]):  # open floor inside three rooms
        cols = []
        for a in ([1.0, 0.0], [0.0, 1.0]):
            env.agent_position = torch.tensor(pos)
            env.step(np.array(a, dtype=np.float32))
            cols.append((env.agent_position - torch.tensor(pos)).numpy() / speed)
        M = np.stack(cols, 1)                      # measured action->displacement map (columns = unit actions)
        s = c.get('shift', {'type': 'none'})
        ang = {'none': 0.0, 'rotate': math.radians(s.get('deg', 0))}.get(s['type'])
        if s['type'] == 'wave':
            ang = math.radians(s['amp_deg']) * math.sin(2 * math.pi * pos[0] / s['period_px'])
        R = np.array([[math.cos(ang), -math.sin(ang)], [math.sin(ang), math.cos(ang)]])
        # rotated unit actions can exceed [-1, 1] per axis only for >45 deg mixes; unit axes never do
        assert np.allclose(M, R, atol=1e-4), (name, pos, M.round(3), R.round(3))
    print(f'{name:10s} speed={speed}  shift={s}  OK')
# clean == stock env, step for step, under the same seed and actions
kw = dict(base, shift={'type': 'none'})
e1, e2 = gym.make(hb.ENV_ID, **kw).unwrapped, gym.make('swm/FourRoomDistractors-v0', **base).unwrapped
o1, _ = e1.reset(seed=3); o2, _ = e2.reset(seed=3)
rng = np.random.default_rng(0)
for t in range(50):
    a = rng.uniform(-1, 1, 2).astype(np.float32)
    o1, *_ = e1.step(a); o2, *_ = e2.step(a)
    assert torch.equal(e1.agent_position, e2.agent_position)
print('clean shift env == stock env over 50 random steps: OK')
'''
run(f'{PY} -c "$CHECK_SHIFT"', env=dict(ENV, CHECK_SHIFT=CHECK_SHIFT, OUT=OUT, SHIFTS=json.dumps(SHIFTS)), cwd=H)

## Process helpers
Long steps run as one background process per GPU (logs in `shift_bench/logs`), while this cell prints progress.

In [ ]:
def launch(cmd, log, gpu=None):
    env = dict(ENV)
    if gpu is not None:
        env['CUDA_VISIBLE_DEVICES'] = str(gpu)
    f = open(log, 'a')
    f.write(f'\n$ {cmd}\n'); f.flush()
    return subprocess.Popen(cmd, shell=True, cwd=H, env=env, stdout=f, stderr=subprocess.STDOUT)

def wait_all(procs, progress=None, every=60):
    t0 = time.time()
    while True:
        codes = [p.poll() for p, _ in procs]
        if progress:
            print(f'[{(time.time() - t0)/60:5.1f} min] ' + progress(), flush=True)
        if all(c is not None for c in codes):
            break
        time.sleep(every)
    bad = [(log, c) for (p, log), c in zip(procs, codes) if c != 0]
    for log, c in bad:
        print(f'--- {log} (exit {c}), last lines ---')
        print(''.join(open(log).readlines()[-40:]))
    assert not bad, 'a worker failed; see the log above'

def plan_progress(conds_, out_dir, seeds, limit=None):
    cfg = json.loads(pathlib.Path(f'{OUT}/conditions.json').read_text())
    def f():
        parts, done_all, total_all = [], 0, 0
        for c in conds_:
            n = cfg[c].get('num_eval', 50) if limit is None else min(limit, cfg[c].get('num_eval', 50))
            done = sum(len(list(pathlib.Path(f'{out_dir}/plan/{c}/seed{s}/tasks').glob('task_*.json'))) for s in seeds)
            parts.append(f'{c} {done}/{n * len(seeds)}'); done_all += done; total_all += n * len(seeds)
        return f'{done_all}/{total_all} tasks | ' + ', '.join(parts)
    return f

## 9. Collect shifted trajectories (CPU, a few minutes)
Same behaviour policy, episode length and seeds as the H-JEPA validation set, for every condition. The trajectories are deterministic, so the "after" notebook gets identical data.

In [ ]:
procs = []
n_par = min(4, len(SHIFTS))
groups = [SHIFTS[i::n_par] for i in range(n_par)]
for i, g in enumerate(groups):
    procs.append((launch(f'{PY} hjepa_shift_bench.py collect --out {OUT} --data-dir {DATA} '
                         f'--conditions {",".join(g)} --episodes {COLLECT_EPISODES}', f'{OUT}/logs/collect_{i}.log'),
                  f'{OUT}/logs/collect_{i}.log'))
wait_all(procs, lambda: f'{len(list(pathlib.Path(DATA).glob("*.done")))}/{len(SHIFTS)} conditions collected', every=20)
for c in SHIFTS:
    print(c, f'{os.path.getsize(f"{DATA}/{c}.h5")/1e6:.0f} MB')

## 10. Part B and C: model error per level and the representation probe (GPU 0, ~10 min)

In [ ]:
for s in SEEDS:
    log = f'{OUT}/logs/model_error_seed{s}.log'
    p = launch(f'{PY} hjepa_shift_bench.py model-error --out {OUT} --data-dir {DATA} --conditions {",".join(SHIFTS)} '
               f'--policy {ckpt(s)} --seed {s} --max-clips {MODEL_ERROR_CLIPS}', log, gpu=GPUS[0])
    wait_all([(p, log)], every=30)
    print(''.join(l for l in open(log) if l.startswith(('[model-error]', '    L', '[probe]'))))

## 11. Smoke test: one short task per condition
Runs into a separate folder (`shift_bench/smoke`) with a 40-step budget, so it never mixes with the real results. It checks the whole planning path and measures time per task.

In [ ]:
if SMOKE:
    SM = f'{OUT}/smoke'
    os.makedirs(SM, exist_ok=True)
    shutil.copy(f'{OUT}/conditions.json', f'{SM}/conditions.json')
    procs = []
    for i, g in enumerate(GPUS):
        log = f'{OUT}/logs/smoke_gpu{g}.log'
        procs.append((launch(f'{PY} hjepa_shift_bench.py plan --out {SM} --conditions {",".join(PLAN_CONDS)} '
                             f'--policy {ckpt(SEEDS[0])} --seed {SEEDS[0]} --limit {len(GPUS)} --shard {i} '
                             f'--num-shards {len(GPUS)} --override eval.eval_budget=40', log, gpu=g), log))
    wait_all(procs, plan_progress(PLAN_CONDS, SM, SEEDS[:1], limit=len(GPUS)), every=30)
    secs = {}
    for c in PLAN_CONDS:
        ts = [json.loads(p.read_text()) for p in pathlib.Path(f'{SM}/plan/{c}/seed{SEEDS[0]}/tasks').glob('*.json')]
        secs[c] = sum(t['seconds'] for t in ts) / len(ts)
        print(f'{c:12s} {secs[c]:6.1f} s per 40-step task')
    full = sum(secs[c] * (210 / 40) * conds[c].get('num_eval', 50) * len(SEEDS) for c in PLAN_CONDS) / len(GPUS)
    print(f'\nrough estimate for the full planning run on {len(GPUS)} GPU(s): {full/3600:.1f} h '
          '(scaled from the 40-step budget to 210; failed tasks run the full budget, so shifts take longer)')

## 12. Part A and D: closed-loop planning under every condition (the long step)
One process per GPU; the tasks are split between them. Each task is seeded on its own (`seed + task index`, as in the repo's evaluation), so the split does not change any result.

In [ ]:
procs = []
for i, g in enumerate(GPUS):
    log = f'{OUT}/logs/plan_gpu{g}.log'
    cmd = ' && '.join(f'{PY} hjepa_shift_bench.py plan --out {OUT} --conditions {",".join(PLAN_CONDS)} '
                      f'--policy {ckpt(s)} --seed {s} --shard {i} --num-shards {len(GPUS)}' for s in SEEDS)
    procs.append((launch(cmd, log, gpu=g), log))
wait_all(procs, plan_progress(PLAN_CONDS, OUT, SEEDS), every=120)

## 13. Summarise

In [ ]:
run(f'{PY} hjepa_shift_bench.py summarize --out {OUT}', env=ENV, cwd=H)
S = json.loads(pathlib.Path(f'{OUT}/summary.json').read_text())

## 14. Results

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

lines = ['# H-JEPA FourRoom dynamics-shift benchmark: before (frozen model, stock planner)', '']
def emit(s=''):
    print(s); lines.append(s)

# ---- A. planning ----
emit('## A. Planning success under each shift (50 fixed tasks)')
emit('| condition | seed | success % | 95% CI | vs clean (paired) | McNemar p | final dist. of failures | steps (successes) |')
emit('|---|---|---|---|---|---|---|---|')
for c in PLAN_CONDS:
    for seed, r in sorted(S['plan'].get(c, {}).items()):
        pv = r.get('paired_vs_clean', {})
        lo, hi = r['wilson95']
        d = f"{pv['delta_success_pts']:+.0f} pts (lost {pv['lost']}, gained {pv['gained']})" if pv else 'reference'
        p = f"{pv['mcnemar_p']:.3f}" if pv else ''
        fd = r['final_distance_failures_mean']
        st = r['mean_steps_success_only']
        emit(f"| {c} | {seed} | {r['success_rate']:.0f} ({r['successes']}/{r['n_tasks']}) | {lo:.0f}-{hi:.0f} | {d} | {p} | "
             f"{'' if fd is None else f'{fd:.0f} px'} | {'' if st is None else f'{st:.0f}'} |")
emit()

# ---- D. headroom ----
if RUN_HEADROOM:
    emit('## D. Planner headroom (same tasks, 90..270 GD iterations vs <=90)')
    emit('| shift | tasks | success 1x | success 3x | gained | lost | McNemar p |')
    emit('|---|---|---|---|---|---|---|')
    for base in HEADROOM_SHIFTS:
        for seed, r in sorted(S['plan'].get(f'{base}_gd3x', {}).items()):
            pv = r.get(f'paired_vs_{base}')
            if pv:
                emit(f"| {base} | {pv['n_common']} | {pv['success_base']:.0f}% | {pv['success_this']:.0f}% | {pv['gained']} | {pv['lost']} | {pv['mcnemar_p']:.3f} |")
    emit()

# ---- planner behaviour ----
emit('## Planner behaviour (1x runs)')
emit('| condition | level | mean GD iters | hit 90-iteration cap | final cost |')
emit('|---|---|---|---|---|')
for c in SHIFTS:
    for seed, r in sorted(S['plan'].get(c, {}).items()):
        for lvl, pl in r['planner'].items():
            fc = pl['final_cost_mean']
            emit(f"| {c} | L{lvl} | {pl['mean_iters']:.1f} | {100*pl['cap_hit_frac']:.0f}% | {'' if fc is None else f'{fc:.3f}'} |")
emit()

# ---- B. model error ----
emit('## B. Model error per level on shifted trajectories')
emit('k-step = autoregressive rollout from 3 true latents with the true actions. rel = MSE / latent variance. '
     '"copy" = predicting no change. "action gap" = error with shuffled actions minus true actions (how much the model uses the action).')
for seed in sorted({s for c in S['model_error'] for s in S['model_error'][c]}):
    ref = S['model_error']['clean'][seed]['levels']
    emit(f'\n**{seed}**\n')
    emit('| condition | level | 1-step loss (x clean) | k | k-step MSE (x clean) | rel | copy MSE | action gap |')
    emit('|---|---|---|---|---|---|---|---|')
    for c in SHIFTS:
        r = S['model_error'][c][seed]
        for lvl, lv in r['levels'].items():
            k = r['horizons'][lvl][-1]
            tf, tf0 = lv['tf_mse']['mean'], ref[lvl]['tf_mse']['mean']
            mk, mk0 = lv[f'mse_k{k}']['mean'], ref[lvl][f'mse_k{k}']['mean']
            emit(f"| {c} | L{lvl} | {tf:.4f} ({tf/tf0:.2f}x) | {k} | {mk:.4f} ({mk/mk0:.2f}x) | {lv[f'rel_mse_k{k}']:.3f} | "
                 f"{lv[f'copy_mse_k{k}']['mean']:.4f} | {lv[f'action_gap_k{k}']:.4f} |")
emit()

# ---- C. probe ----
emit('## C. Representation probe: latent -> agent (x, y), ridge fit on clean')
for seed, pr in sorted(S['probe'].items()):
    for lvl, lv in pr.items():
        emit(f"seed {seed} L{lvl}: " + ', '.join(f"{k}: R2={v['r2']:.3f} ({v['rmse_px']:.1f} px)"
                                              for k, v in lv.items() if isinstance(v, dict)))
emit()

# ---- figures ----
fig, axes = plt.subplots(1, 3, figsize=(17, 4.2))
seed0 = str(f'seed{SEEDS[0]}')
names = [c for c in SHIFTS if seed0 in S['plan'].get(c, {})]
sr = [S['plan'][c][seed0]['success_rate'] for c in names]
ci = np.array([S['plan'][c][seed0]['wilson95'] for c in names])
axes[0].bar(names, sr, yerr=[np.array(sr) - ci[:, 0], ci[:, 1] - np.array(sr)], capsize=4, color='#4C72B0')
axes[0].set_ylabel('success %'); axes[0].set_ylim(0, 105); axes[0].set_title(f'A. planning success ({seed0})')
axes[0].tick_params(axis='x', rotation=30)
if seed0 in S['model_error'].get('clean', {}):
    ref = S['model_error']['clean'][seed0]
    w = 0.8 / max(len(SHIFTS) - 1, 1)
    for j, c in enumerate([c for c in SHIFTS if c != 'clean']):
        r = S['model_error'][c][seed0]
        ratios = [r['levels'][l]['tf_mse']['mean'] / ref['levels'][l]['tf_mse']['mean'] for l in r['levels']]
        axes[1].bar(np.arange(len(ratios)) + j * w, ratios, w, label=c)
    axes[1].axhline(1, color='k', lw=0.8); axes[1].set_xticks(np.arange(len(ratios)) + 0.4 - w / 2)
    axes[1].set_xticklabels([f'L{l}' for l in ref['levels']]); axes[1].set_ylabel('1-step loss / clean')
    axes[1].set_title('B. where the dynamics break'); axes[1].legend(fontsize=8)
    for c in SHIFTS:
        r = S['model_error'][c][seed0]
        ks = r['horizons']['1']
        axes[2].plot(ks, [r['levels']['1'][f'mse_k{k}']['mean'] for k in ks], marker='o', label=c)
    ks = ref['horizons']['1']
    axes[2].plot(ks, [ref['levels']['1'][f'copy_mse_k{k}']['mean'] for k in ks], 'k--', label='copy (clean)')
    axes[2].set_xlabel('rollout steps k (level 1)'); axes[2].set_ylabel('MSE'); axes[2].set_title('B. level-1 rollout error')
    axes[2].legend(fontsize=8)
plt.tight_layout(); plt.savefig(f'{OUT}/figures.png', dpi=130); plt.show()
pathlib.Path(f'{OUT}/report.md').write_text('\n'.join(lines))
print('saved', f'{OUT}/report.md', f'{OUT}/figures.png', f'{OUT}/summary.json')

## How to read this, and what the "after" notebook must show

- **Gap to close (A).** For each shift, the drop from `clean` is what JEPA-TTT or a looped planner has to recover. The paired columns (lost/gained tasks, McNemar p) say whether the drop is real with 50 tasks. A shift that loses only 2 or 3 tasks is too mild to show an improvement; pick shifts with a clear drop for the "after" runs.
- **Which level breaks (B).** A shift that raises the 1-step loss and the k-step error of a level, while `copy` stays the same, is a change in that level's dynamics. That level's predictor is what TTT has to adapt. If only level 1 breaks, level-1 TTT should be enough; if the upper levels break too, their subgoals can become unreachable and they need adapting as well.
- **Encoder unaffected (C).** The probe R² on shifted data should match the clean held-out R². If it does, the failures come from the predictors or the planner, not from what the model sees.
- **Optimisation headroom (D).** If more planner iterations recover tasks on the same model, part of the failure is optimisation, which a looped planner targets. If they don't, and the cap-hit rate is low, the planner already converges and the problem is the model.
- **Keep `shift_bench/`.** The "after" notebook reads `conditions.json`, the shifted trajectories in `data/` and these results, and runs the same script with the adapted model or new planner.